In [ ]:
# Audio Chord Classification (Major vs Minor)
# Author: Gabriela Gomez
# Description: CNN-based classification of chords using spectrograms

# Importing all the required libraries

# Data handling
import numpy as np                   # Numerical operations on arrays
import pandas as pd                  # Tabular data manipulation

# Visualization
import matplotlib.pyplot as plt      # Plotting waveforms and spectrograms

# Audio processing
import librosa                       # Loading audio, computing STFT and features
import librosa.display               # Visualizing audio features

# File handling
import os                            # File and directory operations
from glob import glob                # Find all audio files matching a pattern

# Image processing
import cv2                           # Saving spectrograms as PNG images

# Machine Learning / Deep Learning
import tensorflow as tf              # TensorFlow framework
from tensorflow import keras
from tensorflow.keras.models import Sequential  # Sequential model for CNN
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # Prepare images for training
from keras.callbacks import ModelCheckpoint       # Save the best model during training


In [ ]:
from google.colab import files
files.upload()

In [ ]:
!unzip "archive (1).zip"

In [ ]:
!mv "archive (1).zip" archive.zip
!unzip -o archive.zip

In [ ]:
# Loading a sample audio from the dataset
audio_path = "Audio_Files/Major/Major_6.wav"
data, sr = librosa.load(audio_path)

print(type(data), type(sr))

print ('10 first values of the audio')
print(data[:10])

print ("The shape of the audio: ")
print(data.shape)

print ("Sample rate :" + str(sr))


In [ ]:
# Taking Short-time Fourier transform of the signal
y = librosa.stft(data)
S_db = librosa.amplitude_to_db(np.abs(y), ref=np.max)

S_db_norm = (S_db - S_db.min()) / (S_db.max() - S_db.min())

print(S_db.shape)
print(type(S_db))

In [ ]:
# convert it to "image" values
audioAsImage = (S_db_norm * 255).astype(np.uint8)
print(audioAsImage)
print(audioAsImage.shape)

import cv2
audioAsImage = cv2.resize(audioAsImage, (128, 128))

cv2.imwrite("wave6.png", audioAsImage)

print(audioAsImage.shape)

In [ ]:
major_audio_path = "Audio_Files/Major"
spectrogram_path = "Spectrograms/Major"

# Make sure output folder exists
os.makedirs(spectrogram_path, exist_ok=True)

majorAudioFiles = glob(os.path.join(major_audio_path, "*.wav"))

target_shape = (128, 128)

for file in majorAudioFiles:
    # Load audio
    data, sr = librosa.load(file)

    # Compute STFT (or Mel-spectrogram if you want)
    y = librosa.stft(data)
    S_db = librosa.amplitude_to_db(np.abs(y), ref=np.max)

    S_db_norm = (S_db - S_db.min()) / (S_db.max() - S_db.min())

    # Convert to grayscale image
    audioAsImage = (S_db_norm * 255).astype(np.uint8)

    audioAsImage = cv2.resize(audioAsImage, (target_shape[1], target_shape[0]))

   # Get filename without path
    filename = os.path.basename(file).replace(".wav", ".png")

    # Save PNG in Spectrogram1/Major folder
    save_path = os.path.join(spectrogram_path, filename)
    cv2.imwrite(save_path, audioAsImage)

    print("Saved:", save_path)

print("All Major WAV files converted to grayscale PNGs in Spectrogram1!")

In [ ]:
minor_audio_path = "Audio_Files/Minor"
spectrogram_path = "Spectrograms/Minor"

# Make sure output folder exists
os.makedirs(spectrogram_path, exist_ok=True)

minorAudioFiles = glob(os.path.join(minor_audio_path, "*.wav"))

target_shape = (128, 128)

for file in minorAudioFiles:
    # Load audio
    data, sr = librosa.load(file)

    # Compute STFT (or Mel-spectrogram if you want)
    y = librosa.stft(data)
    S_db = librosa.amplitude_to_db(np.abs(y), ref=np.max)

    S_db_norm = (S_db - S_db.min()) / (S_db.max() - S_db.min())

    # Convert to grayscale image
    audioAsImage = (S_db_norm * 255).astype(np.uint8)

    audioAsImage = cv2.resize(audioAsImage, (target_shape[1], target_shape[0]))

    # Get filename without path
    filename = os.path.basename(file).replace(".wav", ".png")

    # Save PNG in Spectrogram1/Major folder
    save_path = os.path.join(spectrogram_path, filename)
    cv2.imwrite(save_path, audioAsImage)

    print(filename + " - Shape : " + str(audioAsImage.shape))

print("All Minor WAV files converted to grayscale PNGs in Spectrograms/Minor!")

In [ ]:
import os
import shutil
import random
from glob import glob

# Original spectrogram paths
classes = ["Major", "Minor"]
base_spectrogram_path = "Spectrograms"

# Train/validation folders
train_folder = os.path.join(base_spectrogram_path, "train")
val_folder   = os.path.join(base_spectrogram_path, "val")

# Create folders
for cls in classes:
    os.makedirs(os.path.join(train_folder, cls), exist_ok=True)
    os.makedirs(os.path.join(val_folder, cls), exist_ok=True)

# Copy and split images 80/20
for cls in classes:
    img_files = glob(os.path.join(base_spectrogram_path, cls, "*.png"))
    random.shuffle(img_files)
    split_idx = int(0.8 * len(img_files))

    for i, f in enumerate(img_files):
        dest_folder = train_folder if i < split_idx else val_folder
        shutil.copy(f, os.path.join(dest_folder, cls, os.path.basename(f)))

print("Train/Validation folders created!")

In [ ]:
shape = (128, 128)
batchSize = 32

train_folder = "Spectrograms/train"
val_folder   = "Spectrograms/val"

imageGenerator = ImageDataGenerator(rescale=1./255)

train_dataset = imageGenerator.flow_from_directory(
    directory=train_folder,
    target_size=shape,
    color_mode="grayscale",
    class_mode="binary",
    batch_size=batchSize,
    shuffle=True
)

validation_dataset = imageGenerator.flow_from_directory(
    directory=val_folder,
    target_size=shape,
    color_mode="grayscale",
    class_mode="binary",
    batch_size=batchSize,
    shuffle=False
)

In [ ]:


# Get first batch
batch1 = train_dataset[0]

# Get image #6
img = batch1[0][5]   # shape: (128,128,1)
lab = batch1[1][5]   # 0.0 or 1.0

# Remove channel dimension for imshow
img = img.squeeze()   # shape: (128,128)

# Convert label to class name
class_name = "Major" if lab == 0 else "Minor"

print("Image shape:", img.shape)
plt.imshow(img, cmap='gray')  # 🔹 use grayscale colormap
plt.title(class_name)
plt.axis('off')
plt.show()

In [ ]:
train_files = set(train_dataset.filepaths)
val_files = set(validation_dataset.filepaths)

print(len(train_files.intersection(val_files)))

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow import keras
import numpy as np

optimizer = keras.optimizers.Adam(learning_rate=0.0001)

model = keras.models.Sequential([
    keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,1)),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(128, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Flatten(),

    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),

    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

steps_per_epoch = int(np.ceil(train_dataset.samples / batchSize))
validation_steps = int(np.ceil(validation_dataset.samples / batchSize))

best_model_file = "Audio-Major-Minor.h5"

best_model = ModelCheckpoint(
    best_model_file,
    monitor='val_accuracy',
    verbose=1,
    save_best_only=True
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=validation_dataset,
    validation_steps=validation_steps,
    callbacks=[best_model, early_stop]
)

In [ ]:
import matplotlib.pyplot as plt

# Extract history
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc) + 1)

# Plot Accuracy
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs, acc, 'b-', label='Training Accuracy')
plt.plot(epochs, val_acc, 'r-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1,2,2)
plt.plot(epochs, loss, 'b-', label='Training Loss')
plt.plot(epochs, val_loss, 'r-', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
#Install dependencies (run once in Colab)
!pip install librosa sounddevice opencv-python-headless tensorflow

#Import libraries
import librosa
import numpy as np
import cv2
import tensorflow as tf
import IPython.display as ipd
from google.colab import files

#Upload a WAV file
print("Upload a WAV file to test:")
uploaded = files.upload()  # Select your WAV file
testAudio = list(uploaded.keys())[0]  # get filename

#Function to prepare audio for CNN
def prepareAudio(pathForAudio, shape=(128,128)):
    y, sr = librosa.load(pathForAudio)

    # STFT and amplitude to dB
    D = librosa.stft(y)
    S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

    # Normalize 0-1
    S_db_norm = (S_db - S_db.min()) / (S_db.max() - S_db.min())

    # Convert to uint8 grayscale image
    ImageAudio = (S_db_norm * 255).astype(np.uint8)

    # Resize to model input
    resizedImage = cv2.resize(ImageAudio, shape, interpolation=cv2.INTER_AREA)

    # Convert to array, add batch and channel dimension, normalize
    imgResult = resizedImage.reshape((1, shape[0], shape[1], 1)) / 255.0
    return imgResult

#Load your trained CNN (make sure it's uploaded to Colab or saved in session)
best_model_file = "Audio-Major-Minor.h5"  # replace if path differs
model = tf.keras.models.load_model(best_model_file)
print(model.summary())

#Prepare audio image and predict
imageForModel = prepareAudio(testAudio)
resultArray = model.predict(imageForModel, verbose=0)

answer = resultArray[0][0]
predicted_class = "Minor chord" if answer > 0.5 else "Major chord"

#Play audio inline
y, sr = librosa.load(testAudio)
print(f"Prediction: {predicted_class}")
ipd.display(ipd.Audio(y, rate=sr))

In [ ]:
from google.colab import files
files.download("Audio-Major-Minor.h5")